In [1]:
# for i in range(5):
#     print("Hello shubham")

In [31]:
import numpy as np
a = np.array([1,2,3])
print(a)

[1 2 3]


In [32]:
import langchain

In [33]:
from ollama import chat

response = chat(
    model='qwen:latest',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hi there! How can I assist you today?


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen:latest",
    temperature=0
)

inpt = input("Enter your question: ").strip()
response = llm.invoke(
    inpt
)

print(response.content)

Agentic AI refers to the ability of an AI system to make decisions on its own, without explicit instructions from a human user.
Multi-agent systems refer to situations where multiple intelligent entities interact with each other. This can include scenarios such as autonomous vehicles interacting with other vehicles on the road.
Machine learning comes into this picture by allowing the AI system to learn and improve its decision-making abilities over time, through exposure to new data and experiences.


In [35]:
import random

topics = {

"LangChain":[
"chains",
"memory",
"agents",
"prompt templates",
"retrieval",
"tool calling"
],

"LangGraph":[
"state",
"workflow",
"routing",
"checkpointing",
"multi-agent",
"execution"
],

"Ollama":[
"local inference",
"quantization",
"GPU",
"serving",
"model deployment"
],

"RAG":[
"retrieval",
"vector search",
"embeddings",
"context injection",
"chunking"
],

"Agentic AI":[
"planner",
"critic",
"executor",
"supervisor",
"reflection"
]

}

templates=[

"""
Topic: {topic}

Concept: {sub}

Explanation:
{sub} is a major capability of {topic}.
It allows systems to improve reasoning,
manage context,
and produce grounded responses.

Example:
A production AI system can use
{sub}
to increase reliability.

Difficulty:
{level}
""",

"""
Knowledge Area:
{topic}

Focus:
{sub}

Notes:
Modern AI architectures commonly
combine {topic}
with retrieval and orchestration.

Use Case:
Build scalable AI systems.

Level:
{level}
""",

"""
Document:
{topic}

Section:
{sub}

Summary:
{sub}
helps enable adaptive decision making
inside intelligent systems.

Applications:
Education
Healthcare
Finance

Complexity:
{level}
"""
]

levels=[
"Beginner",
"Intermediate",
"Advanced"
]

synthetic=[]

for i in range(100):

    topic=random.choice(
        list(topics.keys())
    )

    sub=random.choice(
        topics[topic]
    )

    synthetic.append(

        random.choice(
            templates
        ).format(

            topic=topic,
            sub=sub,
            level=random.choice(
                levels
            )
        )

    )

with open(
    "knowledge.txt",
    "w"
) as f:

    f.write(
        "\n\n".join(
            synthetic
        )
    )

print(
    "Generated",
    len(synthetic),
    "documents"
)

Generated 100 documents


In [36]:
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import (
RecursiveCharacterTextSplitter
)

from langchain_ollama import (
OllamaEmbeddings
)

from langchain_chroma import (
Chroma
)

loader=TextLoader(
    "knowledge.txt"
)

docs=loader.load()

splitter=RecursiveCharacterTextSplitter(

    chunk_size=700,
    chunk_overlap=100,

    separators=[
        "\n\n",
        "\n",
        ".",
        " "
    ]
)

chunks=splitter.split_documents(
    docs
)

embedding=OllamaEmbeddings(
model="nomic-embed-text"
)

vector_db=Chroma.from_documents(

documents=chunks,

embedding=embedding,

collection_name="synthetic_ai"

)

retriever=vector_db.as_retriever(

search_type="mmr",

search_kwargs={

"k":5

}
)

print(
"Chunks:",
len(chunks)
)

Chunks: 38


In [37]:
def retriever_agent(state):

    docs=(
        retriever.invoke(
            state["question"]
        )
    )

    context=[]

    for i,d in enumerate(docs):

        context.append(

f"""
Source {i+1}

{d.page_content}
"""
        )

    return {

        "context":
        "\n".join(
            context
        )
    }


def reasoning_agent(state):

    prompt=f"""

You are expert AI researcher.

Context:
{state["context"]}

Question:
{state["question"]}

Rules:

Only answer from context.

If uncertain:
say insufficient evidence.

Answer:
"""

    answer=llm.invoke(
        prompt
    )

    return {

        "draft":
        answer.content
    }


def reviewer_agent(state):

    prompt=f"""

Review answer.

Check:

1 Accuracy

2 Grounding

3 Clarity

Answer:

{state["draft"]}

Output improved version.
"""

    out=llm.invoke(
        prompt
    )

    return {

        "final":
        out.content
    }

In [38]:
from typing import TypedDict
from langgraph.graph import StateGraph


class AgentState(TypedDict):

    question:str
    context:str
    draft:str
    final:str


builder = StateGraph(
    AgentState
)

builder.add_node(
    "retrieve",
    retriever_agent
)

builder.add_node(
    "reason",
    reasoning_agent
)

builder.add_node(
    "review",
    reviewer_agent
)

builder.set_entry_point(
    "retrieve"
)

builder.add_edge(
    "retrieve",
    "reason"
)

builder.add_edge(
    "reason",
    "review"
)

builder.set_finish_point(
    "review"
)

app = builder.compile()

print("Graph compiled")

Graph compiled


In [39]:

print(app)

In [40]:
tests = [

"What is LangGraph?",

"Explain RAG",

"How does retrieval work?",

"What is multi-agent AI?"

]

for q in tests:

    print("\nQuestion:", q)

    result = app.invoke({

        "question": q

    })

    print("\nAnswer:\n")

    print(
        result["final"]
    )

    print("\n"+"="*80)


Question: What is LangGraph?

Answer:

The output improved version is not specified in the given information. Therefore, it is not possible to provide an answer to this question.


Question: Explain RAG

Answer:

RAG stands for Rule-Adaptive Graph. It is a framework used in artificial intelligence to reason about the world and make decisions accordingly.
In RAG, rules are used to define the structure of the graph and the relationships between nodes. The rules can be defined by humans or automatically generated based on data available.


Question: How does retrieval work?

Answer:

Retrieval is an essential component of RAG. It enables systems to enhance their reasoning abilities, manage the context effectively, and generate grounded responses.
For instance, a production AI system can leverage retrieval to boost its reliability.


Question: What is multi-agent AI?

Answer:

Multi-agent AI refers to the use of multiple agents in an intelligent system. These agents can be humans, machine

In [41]:
from pathlib import Path

from langchain_community.document_loaders import (
    PyPDFDirectoryLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_ollama import (
    OllamaEmbeddings,
    ChatOllama
)

from langchain_chroma import (
    Chroma
)

from autogen_agentchat.agents import (
    AssistantAgent
)

from autogen_agentchat.teams import (
    RoundRobinGroupChat
)

from autogen_ext.models.ollama import (
    OllamaChatCompletionClient
)

import asyncio

In [42]:
DATA_DIR="./data"

loader = PyPDFDirectoryLoader(
    DATA_DIR
)

docs = loader.load()

print(
    "Pages loaded:",
    len(docs)
)

Unexpected escaped string: u
Unexpected escaped string: i
Unexpected escaped string: F
Unexpected escaped string: F


Pages loaded: 161


In [43]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=1200,

    chunk_overlap=200
)

chunks = splitter.split_documents(
    docs
)

print(
    "Chunks:",
    len(chunks)
)


Chunks: 246


In [44]:
embedding = OllamaEmbeddings(

    model="nomic-embed-text"
)

vector_db = Chroma.from_documents(

    documents=chunks,

    embedding=embedding,

    persist_directory="vector_db"
)

retriever = vector_db.as_retriever(

    search_kwargs={

        "k":5
    }
)

print(
    "Vector DB Ready"
)

Vector DB Ready


In [45]:
def retrieve_context(
    question
):

    docs = retriever.invoke(
        question
    )

    context=[]

    for i,d in enumerate(docs):

        context.append(

f"""
Source {i+1}

{d.page_content}
"""
        )

    return "\n\n".join(
        context
    )

In [46]:
model = OllamaChatCompletionClient(

    model="qwen:latest"
)

print(
    "Qwen Ready"
)

Qwen Ready


In [47]:
retriever_agent = AssistantAgent(

    name="Retriever",

    model_client=model,

    system_message="""

You retrieve evidence.

Only use retrieved context.

Never invent.
"""
)

In [48]:
def retrieve_context(
    question
):

    docs = retriever.invoke(
        question
    )

    context=[]

    for i,d in enumerate(docs):

        context.append(

f"""
Source {i+1}

{d.page_content}
"""
        )

    return "\n\n".join(
        context
    )

In [49]:
retriever_agent = AssistantAgent(

    name="Retriever",

    model_client=model,

    system_message="""

You retrieve evidence.

Only use retrieved context.

Never invent.
"""
)

In [50]:
research_agent = AssistantAgent(

    name="Researcher",

    model_client=model,

    system_message="""

Analyze context.

Explain findings.

Create structured answer.
"""
)

In [51]:
review_agent = AssistantAgent(

    name="Reviewer",

    model_client=model,

    system_message="""

Review answer.

Remove hallucinations.

Improve clarity.
"""
)

In [52]:
team = RoundRobinGroupChat(

    [

        retriever_agent,

        research_agent,

        review_agent

    ],

    max_turns=6
)

print(
    "Team Ready"
)

Team Ready


In [53]:
async def ask(
    question
):

    context = retrieve_context(
        question
    )

    prompt=f"""

Documents:

{context}

Question:

{question}

Rules:

Use evidence.

If missing information,
say insufficient evidence.

"""

    result = await team.run(

        task=prompt
    )

    return result

In [54]:
result = await ask(

"""

Summarize the compliance
requirements.

"""
)

print(result)

messages=[TextMessage(id='5e7f56cd-364a-459c-8da9-83fdd6725538', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 25, 23, 11, 5, 40134, tzinfo=datetime.timezone.utc), content='\n\nDocuments:\n\n\nSource 1\n\nStep 2:   \nClick on “ Show Signer’s certificate ”.\n\n\n\nSource 2\n\nPrinted from www.taxmann.com \nFORM NO. 10-I \n[See rule 11DD] \nCertificate of prescribed authority for the purposes of section 80DDB \n \n \n1.  Name of the Patient   \n \n2.  Address   \n \n3.  Father’s name   \n \n4.  Name and address of the person on whom the patient is dependent \nand his relationship with the patient. \n \n \n5.  Name of the disease or ailment \n(please see rule 11DD) \n \n \n6.  For diseases or ailments mentioned in item (i) of clause (a) of \nsub-rule (1), whether the disability is 40% or more (Please specify \nthe extent). \n \n \n7.  Name, address, registration number and qualification of the \nspecialist issuing the certificate, along with the name

In [ ]:
while True:

    q=input(
        "\nYou:"
    )

    if q=="exit":

        break

    result = await ask(
        q
    )

    print(
        "\nAI:\n"
    )

    print(
        result
    )


AI:

messages=[TextMessage(id='5d5a85c1-87c4-4feb-88a1-298d727694a6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 26, 5, 45, 26, 334145, tzinfo=datetime.timezone.utc), content='\n\nDocuments:\n\n\nSource 1\n\nUser Guide for Cbsi India Pvt Ltd  \nPowered by  \n \nUser Guide for Employee Profile \nEmployee Perspective:  \nOnce employee access the URL provided in the welcome mail with login credentials for accessing the portal, \nbelow show screen appears, where employee needs to update their User Id (by default their employee code) and \nthe password shared in welcome mail. \n \n \nOnce employee clicks on Login button for the first time, after inserting login credentials, system takes them to the \nregistration page, where employee needs to register themselves by updating below mentioned details: \n- Date of Birth \n- Setting of Security Question & Password \n- Setting password of their own choice.\n\n\n\nSource 2\n\nThank You \nThank You\n\n\n\n